# 5) Forecasts in operations — Exercises

**Goals:** derive the cost-minimising quantile, test a rule of thumb for imbalance costs, find the best schedule under steeper penalties, tune the battery's trading rule, and put a price on a better solar forecast.

The setup cell holds lesson 2's demand forecast `REGRESSION`, its honest 2024 errors `ERRORS_2024`, the 2025 demand `ACTUAL`, the solar data `SOLAR`, the prices `PRICE` and `P2025`, the 2024 hourly average price `HOUR_AVG_2024`, and the lecture's `imbalance_cost` and `battery_profit` (with an adjustable trading margin). Power in MW, prices in €/MWh, money in €.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# forecast_data.py lives in the module folder: search upwards so this works from exercise/ and exercise/solutions/
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "forecast_data.py").exists())))
from forecast_data import load_data, market_price, merit_order_price, solar_data, temperature_forecast

warnings.filterwarnings("ignore")
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
dow = df.index.dayofweek
df["daytype"] = np.where(df.holiday | (dow == 6), "sunday_or_holiday", np.where(dow == 5, "saturday", "workday"))
df["hour"], df["month"] = df.index.hour, df.index.month
FORMULA = "demand ~ C(hour):C(daytype) + C(month) + bs(temperature, df=6, lower_bound=-12, upper_bound=40)"
honest = df.assign(temperature=df.temperature_forecast)
REGRESSION = smf.ols(FORMULA, df.loc[:"2024"]).fit().predict(honest)                     # lesson 2's forecast
ERRORS_2024 = (df.demand - smf.ols(FORMULA, df.loc["2023"]).fit().predict(honest)).loc["2024"]
ACTUAL = df.demand.loc["2025"]
SOLAR = solar_data(df)
PRICE = market_price(df.demand - SOLAR.solar)
P2025 = PRICE.loc["2025"]
HOUR_AVG_2024 = PRICE.loc["2024"].groupby(PRICE.loc["2024"].index.hour).mean()


def imbalance_cost(schedule, short=60.0, long=40.0, actual=ACTUAL):
    """Imbalance cost (EUR) of buying `schedule` when `actual` is consumed."""
    gap = actual - schedule.loc[actual.index]
    return float((short * gap.clip(lower=0) + long * (-gap).clip(lower=0)).sum())


def battery_profit(signal, actual=P2025, margin=0.0):
    """The lecture's battery: profit (EUR) and trading days of a daily 4-hour arbitrage driven by `signal`.
    It trades only if 0.85 x forecast dear price exceeds the forecast cheap price by more than `margin` (EUR/MWh)."""
    profit, days = 0.0, 0
    for _, day in pd.DataFrame({"signal": signal, "actual": actual}).groupby(actual.index.date):
        order = day.sort_values("signal")
        cheap, dear = order.index[:4], order.index[-4:]
        if 0.85 * day.loc[dear, "signal"].mean() - day.loc[cheap, "signal"].mean() <= margin:
            continue
        days += 1
        profit += 100 * (0.85 * day.loc[dear, "actual"].sum() - day.loc[cheap, "actual"].sum())
    return profit, days

### Warm-ups

**1. The cost-minimising quantile**

Write `best_quantile(short, long)`: the quantile α* a supplier should buy when a shortfall costs `short` and a surplus `long` €/MWh. Check the lecture's case and a few others.

In [ ]:
def best_quantile(short: float, long: float) -> float:
    """Newsvendor quantile: the share of outcomes that should lie below the amount bought."""
    ...  # YOUR CODE HERE

In [ ]:
assert np.isclose(best_quantile(60, 40), 0.6)
assert np.isclose(best_quantile(40, 40), 0.5), "symmetric costs: buy the median"
assert np.isclose(best_quantile(150, 40), 150 / 190)
assert best_quantile(10, 90) < 0.5, "cheap shortfalls: buy less than the median"

**2. A rule of thumb, and where it breaks**

The lecture found the bill close to ((short + long) / 2) × MAE × hours. Write `rule_of_thumb(schedule)` returning both the exact cost and the rule-of-thumb estimate (MEUR). Test it on the regression, and on the regression with a **bias** of +100 MW and −100 MW added. Why does the rule fail for biased schedules?

In [ ]:
def rule_of_thumb(schedule, short=60.0, long=40.0) -> dict:
    """Exact imbalance cost and the rule-of-thumb estimate, both in MEUR."""
    ...  # YOUR CODE HERE

In [ ]:
thumb = pd.DataFrame({name: rule_of_thumb(REGRESSION + shift) for name, shift in
                      {"regression": 0, "regression + 100 MW": 100, "regression - 100 MW": -100}.items()}).T
thumb["error_%"] = 100 * (thumb.rule_of_thumb_MEUR / thumb.exact_MEUR - 1)
assert abs(thumb.loc["regression", "error_%"]) < 2
assert thumb.loc["regression - 100 MW", "exact_MEUR"] > thumb.loc["regression + 100 MW", "exact_MEUR"]
thumb.round(2)

### Core

**3. Steeper shortfall penalties**

Suppose shortfalls cost 150 €/MWh and surpluses 40 €/MWh. Using the quantiles of `ERRORS_2024`, compute the 2025 cost of buying the regression plus the α-quantile for α in [0.5, 0.6, …, 0.95]. Where is the minimum, how does it compare with `best_quantile(150, 40)`, and how much does it save against the point forecast? Write `steep_penalties()`.

In [ ]:
def steep_penalties(alphas=np.round(np.arange(0.5, 0.951, 0.05), 2)) -> pd.Series:
    """2025 imbalance cost (MEUR) of buying the regression + the alpha-quantile of 2024 errors, short=150, long=40."""
    ...  # YOUR CODE HERE

In [ ]:
steep = steep_penalties()
point = imbalance_cost(REGRESSION, 150, 40) / 1e6
assert abs(steep.idxmin() - best_quantile(150, 40)) <= 0.1, "the minimum lies near the newsvendor quantile"
assert steep.min() < point
print(f"point forecast {point:.2f} MEUR; best alpha on 2025 {steep.idxmin()} -> {steep.min():.2f} MEUR; "
      f"theory alpha* = {best_quantile(150, 40):.3f}")
steep.round(2)

**4. How demanding should the battery be?**

The lecture's battery traded whenever 0.85 × the forecast dear price exceeded the forecast cheap price. Require a **margin**: trade only if that difference exceeds m €/MWh, for m in [−5, 0, 5, 10, 20]. Use the tie-broken fundamental forecast. Report profit and trading days, and compare with perfect foresight at the same margins. Write `margin_sweep(margins)`.

In [ ]:
FUNDAMENTAL = pd.Series(merit_order_price(REGRESSION - SOLAR.solar_forecast), index=df.index).loc["2025"]
TIE_BROKEN = FUNDAMENTAL + 1e-6 * pd.Series(HOUR_AVG_2024.loc[P2025.index.hour].to_numpy(), index=P2025.index)


def margin_sweep(margins=(-5, 0, 5, 10, 20)) -> pd.DataFrame:
    """Profit (MEUR) and trading days for each trading margin, forecast-driven and with perfect foresight."""
    ...  # YOUR CODE HERE

In [ ]:
sweep = margin_sweep()
assert np.isclose(sweep.loc[0, "forecast_profit_MEUR"], 1.29, atol=0.01), "the lecture's 1.29 MEUR at margin 0"
assert sweep.forecast_days.is_monotonic_decreasing
sweep.round(2)

**5. Interpretation**

Why does the rule of thumb fail for biased schedules, and in which direction? How did the steeper penalty move the best quantile, and was the theory right? What does the margin sweep say about how a battery should use an imperfect forecast?

*Your answer here.*

### Challenge

**6. What is a better solar forecast worth to the battery?**

Scale the solar forecast error: solar forecast_s = solar + s · (solar forecast − solar), for s in [0, 0.5, 1, 1.5]. For each s, build the fundamental price forecast from `REGRESSION` and the scaled solar forecast, break ties as in task 4, and report the price-forecast MAE for 2025 and the battery profit (margin 0). Write `solar_value(scales)`.

In [ ]:
def solar_value(scales=(0, 0.5, 1, 1.5)) -> pd.DataFrame:
    """Price-forecast MAE (EUR/MWh) and battery profit (MEUR) for scaled solar-forecast errors."""
    ...  # YOUR CODE HERE

In [ ]:
value = solar_value()
assert np.isclose(value.loc[1, "battery_profit_MEUR"], 1.29, atol=0.01)
assert value.loc[0, "battery_profit_MEUR"] > value.loc[1.5, "battery_profit_MEUR"]
value.round(3)

**7. Interpretation**

How much would a perfect solar forecast be worth to this battery, and how much does a 50 % worse one cost? Why does even a perfect solar forecast leave the battery far below perfect foresight?

*Your answer here.*